In [1]:
# ============================================================
# Cell 0: GPU Check
# ============================================================
!nvidia-smi

Sat Mar  7 06:23:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ============================================================
# Cell 1: Setup — Clone, Install, GPU Auto-Detect
# ============================================================
# Stage A: Strong honest FEVER baseline on A100
# No symbolic constraints yet — pure neural cross-entropy
import os, subprocess, sys, time

t0 = time.time()

# ── Clone / update repo ──
REPO = "/content/nst"
if not os.path.exists(REPO):
    print("Cloning repo...")
    subprocess.run(["git", "clone",
        "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git",
        REPO], check=True)
    print("Repo cloned")
else:
    r = subprocess.run(["git", "pull", "--ff-only"],
                       capture_output=True, text=True, cwd=REPO)
    print(f"git pull: {r.stdout.strip()}")

os.chdir(REPO)
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

# ── Install deps (keep Colab's pre-installed PyTorch + CUDA) ──
print("\nInstalling dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.40", "datasets>=2.18",
    "sentencepiece>=0.1.99", "protobuf>=4.0",
    "pyyaml", "scikit-learn", "rank-bm25==0.2.2",
    "accelerate", "peft>=0.7.0", "tiktoken",
    "fsspec>=2023.6,<2025", "huggingface_hub>=0.21,<1.0",
    "scipy>=1.9.0"],
    check=True, capture_output=True, text=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
    capture_output=True, text=True)
print("pip done")

# ── Verify core installs ──
import torch, transformers, peft
print(f"\nPyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

# ════════════════════════════════════════════════════════════
#  GPU Auto-Detection — A100 Optimized
# ════════════════════════════════════════════════════════════
import torch

BASELINE_CONFIG = "configs/fever_baseline_a100.yaml"
GPU_OVERRIDES = {}

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = props.total_memory / 1e9
    cc = (props.major, props.minor)

    supports_bf16 = cc >= (8, 0)
    supports_tf32 = cc >= (8, 0)

    print(f"\nGPU          : {gpu_name}")
    print(f"VRAM         : {vram_gb:.1f} GB")
    print(f"Compute Cap  : {cc[0]}.{cc[1]}")

    # ── Tier-based batch size ──
    if vram_gb >= 70:       # H100 80GB / A100 80GB
        TIER, BS, GA, WORKERS = "H100/A100-80GB", 48, 2, 4
    elif vram_gb >= 35:     # A100 40GB (Colab)
        TIER, BS, GA, WORKERS = "A100-40GB", 32, 2, 4
    elif vram_gb >= 20:     # L4 24GB
        TIER, BS, GA, WORKERS = "L4-24GB", 24, 2, 2
    else:                   # T4 16GB
        TIER, BS, GA, WORKERS = "T4-16GB", 16, 2, 2
        # T4 needs base model, not large
        BASELINE_CONFIG = "configs/fever_gold_neural.yaml"

    # Build overrides that get merged into the config
    GPU_OVERRIDES = {
        "train": {
            "batch_size":       BS,
            "grad_accum_steps": GA,
            "bf16":             supports_bf16,
            "fp16":             not supports_bf16,
            "tf32":             supports_tf32,
            "benchmark":        True,
            "fused_optimizer":  True,
            "num_workers":      WORKERS,
        }
    }

    # Enable GPU optimizations NOW
    if supports_tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    eff_bs = BS * GA
    prec = "BF16" if supports_bf16 else "FP16"
    # Estimate runtime: ~145K train, 3 epochs
    steps_est = 3 * (145000 // BS) // GA
    sec_per_step = 0.4 if vram_gb >= 35 else 0.8
    est_min = steps_est * sec_per_step / 60

    print(f"\n{'='*60}")
    print(f"  CONFIGURED FOR: {TIER}")
    print(f"  Config    : {BASELINE_CONFIG}")
    print(f"  Batch     : {BS} x {GA} = {eff_bs} effective")
    print(f"  Precision : {prec}")
    print(f"  TF32      : {'ON' if supports_tf32 else 'OFF'}")
    print(f"  Workers   : {WORKERS}")
    print(f"  Est steps : ~{steps_est}")
    print(f"  Est time  : ~{est_min:.0f} min")
    print(f"{'='*60}")
else:
    print("\nWARNING: No CUDA GPU detected!")
    print("Go to: Runtime > Change runtime type > GPU")

print(f"\nSetup done ({time.time()-t0:.0f}s)")

: 

In [4]:
# ============================================================
# Cell 2: Build FEVER Wiki Cache (one-time, ~4 min)
# ============================================================
# The wiki cache maps page titles → sentences for gold evidence lookup.
# Built once as SQLite (~15MB), reused across all runs.
import os, sys, time

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

# Clear stale modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_wiki_cache import WikiCache

cache_path = "data/fever_wiki.db"
if os.path.exists(cache_path):
    cache = WikiCache(cache_path)
    n = len(cache)
    print(f"Wiki cache exists: {n} pages, "
          f"{os.path.getsize(cache_path)/1024/1024:.1f} MB")
    cache.close()
else:
    print("Building wiki cache from HuggingFace (downloads ~300MB, ~4 min)...")
    t0 = time.time()
    import subprocess
    r = subprocess.run([sys.executable, "main.py", "build-fever-wiki-cache"],
                       capture_output=True, text=True, timeout=600)
    print(r.stdout[-2000:] if r.stdout else "(no stdout)")
    if r.returncode != 0:
        print(f"STDERR: {r.stderr[-1000:]}")
        raise RuntimeError("Wiki cache build failed!")
    print(f"Wiki cache built in {time.time()-t0:.0f}s")

# Quick verification
if os.path.exists(cache_path):
    cache = WikiCache(cache_path)
    n = len(cache)
    sample_titles = cache.titles()[:3]
    for t in sample_titles:
        sents = cache.lookup(t)
        print(f"  '{t}': {len(sents)} sentences")
    cache.close()
    print(f"\nWiki cache verified: {n} pages")
else:
    raise FileNotFoundError("Wiki cache not found after build!")

Building wiki cache from HuggingFace FEVER dataset...
This downloads ~300MB and takes ~4 minutes. One-time only.

  Built: 14363/14533 pages (170 missing) in 363.6s
  Cache: data/fever_wiki.db (24.2 MB)


Wiki cache built in 365s
  '"Heroes"_-LRB-David_Bowie_album-RRB-': 9 sentences
  ''Til_Death': 4 sentences
  '...More_Unchartered_Heights_of_Disgrace': 13 sentences
Wiki cache verified: 14363 pages


In [5]:
# ============================================================
# Cell 3: Data Sanity Check — Verify FEVER loads correctly
# ============================================================
# Quick check: load splits, print stats, verify evidence text exists.
# This catches broken wiki cache, missing data, bad labels BEFORE training.
import os, sys, logging

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

from data.fever_dataset import load_fever_splits, print_fever_stats

# Load small sample to verify pipeline (not training, just checking)
splits = load_fever_splits(max_train=500, max_dev=200, dev_test_ratio=0.1, seed=42)
print_fever_stats(splits)

# Verify evidence is resolved (not just titles)
train_items = splits["train"]
n_with_evidence = sum(1 for it in train_items if len(it["gold_evidence_text"]) > 30)
pct = 100 * n_with_evidence / len(train_items)
print(f"\n  Evidence quality: {n_with_evidence}/{len(train_items)} ({pct:.0f}%) "
      f"have >30 char evidence text")

# Show a sample
for i in [0, 1, 2]:
    it = train_items[i]
    ev = it["gold_evidence_text"][:120] + ("..." if len(it["gold_evidence_text"]) > 120 else "")
    print(f"\n  [{i}] Claim: {it['claim'][:80]}")
    print(f"      Label: {it['label']}")
    print(f"      Evidence: {ev}")

if pct > 60:
    print(f"\n  Data check PASSED — evidence text is resolving correctly")
else:
    print(f"\n  WARNING: Only {pct:.0f}% have real evidence text!")
    print(f"  Check wiki cache. Low evidence resolution will hurt accuracy.")

  SMOKE TEST: DeBERTa-v3-base, 200 train, 100 dev, 1 epoch



train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 200 examples (105 with evidence text, 95 without)
fever_dataset |   dev: 100 examples (47 with evidence text, 53 without)
fever_dataset |   train hash: 398028c26c5ec3e7
fever_dataset |   dev hash: e3b09c2bfa26d269
train_fever | Using GOLD EVIDENCE mode (Setting A

  FEVER Dataset Statistics

  train: 200 examples
    With gold evidence: 105 (52.5%)
    Label distribution:
      SUPPORTS                129  (64.5%)
      REFUTES                  16  (8.0%)
      NOT ENOUGH INFO          55  (27.5%)
    Split hash: 398028c26c5ec3e7

  dev: 100 examples
    With gold evidence: 47 (47.0%)
    Label distribution:
      SUPPORTS                 46  (46.0%)
      REFUTES                  30  (30.0%)
      NOT ENOUGH INFO          24  (24.0%)
    Split hash: e3b09c2bfa26d269


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

  Epoch 1/1: loss=1.1638 constraint=0.0000 | dev_acc=0.3300 ECE=0.0340

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


  Learned temperature: T = 2.3206

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
train_fever | Report saved to outputs_fever_gold_smoke/report.json


  Label Accuracy (GOLD evidence): 0.3300
  ECE: 0.0340
  Brier: 0.6808
    SUPPORTS: acc=0.0000 (n=46)
    REFUTES: acc=0.3000 (n=30)
    NOT ENOUGH INFO: acc=1.0000 (n=24)

  Training complete in 8.5s
  Best dev accuracy: 0.3300
  Output: outputs_fever_gold_smoke

SMOKE TEST RESULTS (72s):
  dev_accuracy : 0.33
  dev_ece      : 0.033954
  nan_abort    : False

 Smoke test passed! Pipeline working correctly.


In [6]:
# ============================================================
# Cell 4: Smoke Test — Quick 200-example run (validates full pipeline)
# ============================================================
# Runs 1 epoch on 200 train / 100 dev to validate:
# - Model builds and LoRA activates
# - Tokenization works
# - Forward/backward pass runs
# - Evaluation loop completes
# - No NaN, no crashes
# Takes 1-2 minutes. Catches issues before the long run.
import os, sys, time, logging, gc, torch

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  SMOKE TEST: 200 train, 100 dev, 1 epoch")
print("  Validates: model build, LoRA, forward/backward, eval")
print("=" * 60 + "\n")

# Use the same config as full run but with tiny data + 1 epoch
SMOKE_OVERRIDES = {
    "data": {"max_train": 200, "max_dev": 100},
    "train": {
        "epochs": 1,
        "eval_strategy": "epoch",
        "patience": 99,
    },
    "io": {"out_dir": "outputs_smoke"},
}
# Merge GPU overrides
if GPU_OVERRIDES:
    for k, v in GPU_OVERRIDES.items():
        if k in SMOKE_OVERRIDES:
            SMOKE_OVERRIDES[k].update(v)
        else:
            SMOKE_OVERRIDES[k] = dict(v)
# Override batch to be small for smoke
SMOKE_OVERRIDES["train"]["batch_size"] = min(GPU_OVERRIDES.get("train", {}).get("batch_size", 16), 16)
SMOKE_OVERRIDES["train"]["grad_accum_steps"] = 1

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results = train_fever_nst(BASELINE_CONFIG, config_overrides=SMOKE_OVERRIDES)
elapsed = time.time() - t0

print("\n" + "=" * 60)
print(f"SMOKE TEST RESULTS ({elapsed:.0f}s):")
print("=" * 60)
dev = results.get("dev", {})
print(f"  use_lora         : {results.get('use_lora', '?')}")
print(f"  trainable_params : {results.get('trainable_params_M', '?')}M")
print(f"  total_params     : {results.get('total_params_M', '?')}M")
print(f"  dev_accuracy     : {dev.get('accuracy', 'N/A')}")
print(f"  nan_abort        : {results.get('nan_abort', False)}")

# Validate LoRA is truly active
tp = results.get("trainable_params_M", 0)
total = results.get("total_params_M", 0)
if total > 0 and tp / total > 0.5:
    print(f"\n  WARNING: {tp}M/{total}M trainable = {100*tp/total:.0f}%")
    print(f"  LoRA may not be active! Check model config.")
elif total > 0:
    print(f"\n  LoRA check: {tp}M/{total}M = {100*tp/total:.1f}% trainable — OK")

if results.get("nan_abort", False):
    print("\n  SMOKE TEST FAILED — NaN detected!")
else:
    print("\n  Smoke test PASSED — ready for full run")

# Cleanup smoke outputs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  NEURAL BASELINE (Setting A: Gold Evidence)
  DeBERTa-v3-base, Full FEVER train (~145K), 3 epochs
  batch=16, grad_accum=2 (eff. batch=32)
  Eval: every 500 steps on 2K dev subset



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (77591 with evidence text, 67858 without)
fever_dataset |   dev: 19998 examples (8441 with evidence text, 11557 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 77591 (53.3%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 7581 (42.1%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 860 (43.0%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: microsoft/deberta-v3-base (184.4M params)
train_fever | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever | Periodic eval uses dev subset: 2000/17998 examples


...

  Learned temperature: T = 1.1094

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Label Accuracy (GOLD evidence): 0.8252
  ECE: 0.0286
  Brier: 0.2445
    SUPPORTS: acc=0.8814 (n=6014)
    REFUTES: acc=0.8266 (n=5969)
    NOT ENOUGH INFO: acc=0.7676 (n=6015)

────────────────────────────────────────
  Final evaluation on held-out dev_test
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Label Accuracy (GOLD evidence): 0.8255
  ECE: 0.0384
  Brier: 0.2453
    SUPPORTS: acc=0.8788 (n=652)
    REFUTES: acc=0.8364 (n=697)
    NOT ENOUGH INFO: acc=0.7604 (n=651)

  Training complete in 4405.4s
  Best dev accuracy: 0.8430
  Output: outputs_fever_gold_neural

NEURAL BASELINE RESULTS (81.4 min):
  dev accuracy      : 0.8252
  dev ECE           : 0.028639
  dev Brier         : 0.24446
  dev_test accuracy : 0.8255
  dev_test ECE      : 0.03835
  temperature       : 1.1094
  best_dev_acc      : 0.843

Per-label:
    SUPPORTS: acc=0.8814 (n=6014)
    REFUTES: acc=0.8266 (n=5969)
    NOT ENOUGH INFO: acc=0.7676 (n=6015)

Neural baseline done in 81.4 min


In [7]:
# ============================================================
# Cell 5: FULL FEVER BASELINE — DeBERTa-v3-large + LoRA
# ============================================================
# Stage A: Strongest honest neural baseline
#
# - DeBERTa-v3-large (304M params) with LoRA r=16 (~2.5M trainable)
# - Gold evidence (Setting A: oracle evidence provided)
# - Full FEVER train (~145K), eval on dev (~17K), final on dev_test (~2K)
# - BF16 mixed precision, fused AdamW, TF32
# - Cosine LR schedule with warmup
# - Early stopping on dev accuracy (patience=5)
# - Post-hoc temperature scaling
#
# This is the real run. Takes ~25-40 min on A100-40GB.
import os, sys, time, logging, json, gc, torch

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

# Show what we're about to run
import yaml
with open(BASELINE_CONFIG) as f:
    cfg = yaml.safe_load(f)
model_name = cfg.get("model", {}).get("name", "?")
use_lora = cfg.get("model", {}).get("use_lora", False)
lora_r = cfg.get("model", {}).get("lora_rank", 0)
ov_bs = GPU_OVERRIDES.get("train", {}).get("batch_size", cfg.get("train", {}).get("batch_size", "?"))
ov_ga = GPU_OVERRIDES.get("train", {}).get("grad_accum_steps", cfg.get("train", {}).get("grad_accum_steps", "?"))
ov_prec = "BF16" if GPU_OVERRIDES.get("train", {}).get("bf16") else "FP16"

print("=" * 65)
print("  STAGE A: FULL FEVER NEURAL BASELINE")
print("  (Gold Evidence, Setting A)")
print("=" * 65)
print(f"  Model       : {model_name}")
print(f"  LoRA        : r={lora_r}" if use_lora else "  LoRA        : OFF (full fine-tune)")
print(f"  Config      : {BASELINE_CONFIG}")
print(f"  Batch       : {ov_bs} x {ov_ga} = {ov_bs*ov_ga if isinstance(ov_bs, int) and isinstance(ov_ga, int) else '?'} effective")
print(f"  Precision   : {ov_prec}")
print(f"  Epochs      : {cfg.get('train', {}).get('epochs', '?')}")
print(f"  Eval every  : {cfg.get('train', {}).get('eval_every_steps', '?')} steps")
print("=" * 65 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_baseline = train_fever_nst(BASELINE_CONFIG, config_overrides=GPU_OVERRIDES)
elapsed = time.time() - t0

print("\n" + "=" * 65)
print(f"BASELINE RESULTS ({elapsed/60:.1f} min):")
print("=" * 65)
print(f"  Model             : {results_baseline.get('model_name', '?')}")
print(f"  LoRA              : r={results_baseline.get('lora_rank', 0)}")
print(f"  Trainable params  : {results_baseline.get('trainable_params_M', '?')}M / "
      f"{results_baseline.get('total_params_M', '?')}M")
print(f"  Effective batch   : {results_baseline.get('effective_batch_size', '?')}")
print(f"  Precision         : {results_baseline.get('precision', '?')}")
print()

dev = results_baseline.get("dev", {})
dev_test = results_baseline.get("dev_test", {})
print(f"  Dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE           : {dev.get('ece', 'N/A')}")
print(f"  Dev Brier         : {dev.get('brier', 'N/A')}")
if dev_test:
    print(f"  DevTest accuracy  : {dev_test.get('accuracy', 'N/A')}  (held-out, FINAL)")
    print(f"  DevTest ECE       : {dev_test.get('ece', 'N/A')}")
print(f"  Temperature       : {results_baseline.get('temperature', 'N/A')}")
print(f"  Best dev acc      : {results_baseline.get('best_dev_acc', 'N/A')}")

print(f"\n  Per-label breakdown (dev):")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: acc={stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Save results
with open("results_baseline.json", "w") as f:
    json.dump(results_baseline, f, indent=2, default=str)
print(f"\n  Saved to results_baseline.json")
print(f"  Checkpoint in: {results_baseline.get('io', {}).get('out_dir', 'outputs_fever_baseline')}/ckpt/")
print(f"\n  Baseline done in {elapsed/60:.1f} min")

train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  NST SOFT CONSTRAINTS (Setting A: Gold Evidence)
  CE + fixed lambda=0.1 x constraint_loss
  5 constraints: date, number, negation, entity, empty



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (77591 with evidence text, 67858 without)
fever_dataset |   dev: 19998 examples (8441 with evidence text, 11557 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 77591 (53.3%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 7581 (42.1%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 860 (43.0%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: microsoft/deberta-v3-base (184.4M params)
train_fever | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever | Periodic eval uses dev subset: 2000/17998 examples


...

  Step 4500: loss=0.3039 λ=0.1000 | dev_acc=0.8415 ECE=0.0185


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Epoch 1/3: loss=0.5839 constraint=0.4076


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

: 

In [1]:
# ============================================================
# Cell 6: Results Summary
# ============================================================
import json, os

os.chdir("/content/nst")

print("=" * 65)
print("  FEVER BASELINE RESULTS — Setting A: Gold Evidence")
print("  Honest evaluation, no data leakage, held-out dev_test")
print("=" * 65)

if os.path.exists("results_baseline.json"):
    with open("results_baseline.json") as f:
        r = json.load(f)

    dev = r.get("dev", {})
    dt = r.get("dev_test", {})

    print(f"\n  Model           : {r.get('model_name', '?')}")
    print(f"  LoRA            : r={r.get('lora_rank', 0)}")
    print(f"  Trainable       : {r.get('trainable_params_M', '?')}M / {r.get('total_params_M', '?')}M")
    print(f"  Batch           : {r.get('effective_batch_size', '?')} effective")
    print(f"  Precision       : {r.get('precision', '?')}")
    print(f"  Training time   : {r.get('elapsed_s', 0)/60:.1f} min")
    print(f"  NaN abort       : {r.get('nan_abort', False)}")

    print(f"\n  {'Metric':<20} {'Dev':<12} {'DevTest':<12}")
    print(f"  {'-'*44}")
    print(f"  {'Accuracy':<20} {dev.get('accuracy', 0):<12.4f} {dt.get('accuracy', 0) if dt else '--':<12}")
    print(f"  {'ECE':<20} {dev.get('ece', 0):<12.4f} {dt.get('ece', 0) if dt else '--':<12}")
    print(f"  {'Brier':<20} {dev.get('brier', 0):<12.4f} {dt.get('brier', 0) if dt else '--':<12}")

    print(f"\n  Per-label (dev):")
    for label, stats in dev.get("per_label", {}).items():
        print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

    # Training curve
    log = r.get("train_log", [])
    if log:
        print(f"\n  Training trajectory ({len(log)} checkpoints):")
        for entry in log[-5:]:  # Last 5 entries
            step = entry.get("global_step", entry.get("epoch", "?"))
            acc = entry.get("dev_accuracy", 0)
            loss = entry.get("train_loss", 0)
            print(f"    step/epoch {step}: loss={loss:.4f} dev_acc={acc:.4f}")

    print(f"\n  Temperature     : {r.get('temperature', 'N/A')}")
    print(f"  Best dev acc    : {r.get('best_dev_acc', 0):.4f}")

    # Integrity note
    print(f"\n{'='*65}")
    print("  INTEGRITY NOTES:")
    print("  - Dev accuracy used for early stopping / model selection")
    print("  - DevTest is held-out 10% of labelled_dev, NEVER used for tuning")
    print("  - Gold evidence = oracle Setting A (evidence provided, not retrieved)")
    print("  - No symbolic constraints in this baseline")
    print(f"{'='*65}")
else:
    print("\n  No results found. Run Cell 5 first.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/nst'

In [ ]:
# ============================================================
# Cell 7: (Optional) Save Results to Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil, datetime, os, glob

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# Collect outputs
os.makedirs("all_fever_outputs", exist_ok=True)
for d in sorted(glob.glob("outputs_fever_*")):
    dst = os.path.join("all_fever_outputs", os.path.basename(d))
    if not os.path.exists(dst):
        shutil.copytree(d, dst)
for f in glob.glob("results_*.json"):
    shutil.copy(f, "all_fever_outputs/")

archive = shutil.make_archive(f"nst_fever_baseline_{timestamp}", "zip", "all_fever_outputs")
dst_dir = "/content/drive/MyDrive/"
shutil.copy(archive, dst_dir)
print(f"Saved: {dst_dir}{os.path.basename(archive)}")
print("\nDone! Remember: Runtime > Disconnect and delete runtime (save credits)")